# Session 03: Functions, Exceptions, Files and JSON

**Notebook 1 of 2: Concepts** · used during the live session and for revision afterwards (about 60 minutes on your own)

In Session 02 you transformed data that was already in memory. Real data arrives from outside: files, APIs, other teams' exports. It's often late, malformed or missing. This session covers the tools for writing code that survives contact with that data: well-designed functions, exceptions used deliberately, safe file handling and JSON.

By the end of this notebook you will be able to:

1. **Design** functions with positional, keyword-only and default arguments, and **avoid** the mutable-default trap.
2. **Annotate** functions with type hints, and **explain** why Python does not enforce them at run time.
3. **Raise**, **catch** and **chain** exceptions, and **read** a multi-frame traceback.
4. **Use** `pathlib` and context managers for file I/O, and **explain** what `with` guarantees.
5. **Parse**, **validate by hand** and **serialise** JSON, handling malformed input without crashing the whole job.

> **Kernel:** the course environment's `.venv`, as before. Everything here uses only the standard library. Examples that touch files work in a temporary folder, which is deleted afterwards, so they leave nothing behind.

## The engineering problem

The Orbital Logistics (fictional) on-call summary from Session 02 now runs every night against real export files. Last week it failed twice:

1. **At 02:00 on Tuesday**, one export file had been cut off halfway through. The job crashed on that file with a `JSONDecodeError`, and nobody got a report, not even for the four files that were fine.
2. **On Wednesday** the report included tickets from Tuesday's run. A helper function was "remembering" data between calls, although nobody wrote any caching code.

Both are design problems, not typos. The first is about **where exceptions are handled**. The second is a Python rule about **when default arguments are evaluated**, a direct consequence of the names-and-objects model from Session 02.

## Part 1: Functions and the call

You know functions. Three Python specifics are worth stating up front:

- A function without a `return` (or with a bare `return`) returns `None`. There is no `void`.
- Returning "several values" means returning one tuple, which the caller unpacks.
- Functions are objects. You can pass them as arguments (you did with `key=` in Session 02), store them in dicts, and return them.

![What happens when you call a function](../../../assets/diagrams/function-call-flow.svg)

**How this works**

- Calling `summarise(exports, sla_hours=SLA)` creates a new **frame** on the call stack, a private namespace for this one call. It disappears when the call returns.
- Parameters are bound to the objects the caller passed, exactly like `tickets = exports` (Session 02, Part 1). Nothing is copied, so a function that mutates `tickets` mutates the caller's list.
- Names created inside the function (`open_tickets`) belong to its frame. When the frame is discarded, the names go with it, and any object with no names left is freed.
- `return` hands one object back to the caller, which binds it to `count`. Here that's an `int`, but it could be a tuple of several values, a dict, or `None`.
- The stack is also how exceptions travel: when something goes wrong deep inside, Python walks back up these frames looking for a handler (Part 5).

In [ ]:
def open_and_breached(tickets, sla_hours):
    """Return (open_count, breached_count) for a list of normalised tickets."""
    open_tickets = [t for t in tickets if t["status"] != "closed"]
    breached = [t for t in open_tickets if t["age_hours"] > sla_hours[t["priority"]]]
    return len(open_tickets), len(breached)


tickets = [
    {"id": "INC-1", "status": "open", "priority": "P1", "age_hours": 6},
    {"id": "INC-2", "status": "closed", "priority": "P2", "age_hours": 40},
    {"id": "INC-3", "status": "open", "priority": "P2", "age_hours": 3},
]
SLA_HOURS = {"P1": 4, "P2": 24}

open_count, breached_count = open_and_breached(tickets, SLA_HOURS)
print(open_count, breached_count)
print(open_and_breached.__doc__)

The triple-quoted string under `def` is a **docstring**. Editors show it on hover, and `help()` prints it. Write one for every function someone else will call: what it does, what it returns, and what it raises.

**Functions are objects.** A dict of functions replaces a long `if`/`elif` chain when you dispatch on a value:

In [ ]:
def format_text(count):
    return f"{count} open tickets"


def format_markdown(count):
    return f"**{count}** open tickets"


FORMATTERS = {"text": format_text, "markdown": format_markdown}

for output_format in ["text", "markdown"]:
    formatter = FORMATTERS[output_format]   # a function, not yet called
    print(formatter(open_count))            # call it

## Part 2: Arguments

Python's argument system is richer than Java's or C#'s, and most of its features exist to make **call sites** readable.

In [ ]:
def find_tickets(tickets, team, *, status="open", min_age_hours=0):
    """Tickets for `team` with the given status, at least `min_age_hours` old."""
    return [
        t["id"]
        for t in tickets
        if t["team"] == team and t["status"] == status and t["age_hours"] >= min_age_hours
    ]


TEAM_TICKETS = [
    {"id": "INC-1", "team": "platform", "status": "open", "age_hours": 30},
    {"id": "INC-2", "team": "platform", "status": "closed", "age_hours": 50},
    {"id": "INC-3", "team": "database", "status": "open", "age_hours": 2},
]

print(find_tickets(TEAM_TICKETS, "platform"))                        # defaults
print(find_tickets(TEAM_TICKETS, "platform", status="closed"))       # keyword argument
print(find_tickets(team="database", tickets=TEAM_TICKETS))          # any parameter by name

The bare `*` in the signature makes every parameter after it **keyword-only**. Callers must write `status="closed"`; they can't pass it by position.

**Predict:** what happens with `find_tickets(TEAM_TICKETS, "platform", "closed")`?

In [ ]:
try:
    find_tickets(TEAM_TICKETS, "platform", "closed")
except TypeError as error:
    print("TypeError:", error)

**Explain:** the call fails immediately, with a message that names the problem. That's the point. `find_tickets(tickets, "platform", "closed", 24)` would be unreadable: which of those is the status, and which the age? Keyword-only arguments force call sites to say what they mean, and let you reorder or add optional parameters later without breaking callers.

**Rule of thumb:** one or two obvious positional parameters (the data being worked on), then `*`, then options.

| Feature | Java | C# | TypeScript | Python |
|---|---|---|---|---|
| Default values | Overloads | `int x = 0` | `x = 0` | `x=0` |
| Pass by name | No | `Find(team: "x")` | Options object `{ team }` | `find(team="x")` |
| Force naming | No | No | Options object | `*` in the signature |
| Variable arguments | `String... xs` | `params string[] xs` | `...xs` | `*args` (tuple), `**kwargs` (dict) |

You'll meet `*args` and `**kwargs` when reading library code and wrappers: `def log_call(*args, **kwargs)` accepts any positional arguments as a tuple and any keyword arguments as a dict. Use them sparingly in your own code, because they hide what a function actually accepts. (A `/` in a signature, as in `def f(x, /)`, means the parameters before it are *positional-only*. You'll see it in the standard library's documentation.)

### The mutable default trap

Here is Wednesday's bug from the engineering problem. This helper collects the tickets to include in a report.

**Predict:** what does the second call print?

In [ ]:
def collect_breaches(tickets, sla_hours, found=[]):
    for ticket in tickets:
        if ticket["age_hours"] > sla_hours[ticket["priority"]]:
            found.append(ticket["id"])
    return found


tuesday = [{"id": "INC-1", "priority": "P1", "age_hours": 9}]
wednesday = [{"id": "INC-7", "priority": "P1", "age_hours": 5}]

print("Tuesday:  ", collect_breaches(tuesday, SLA_HOURS))
print("Wednesday:", collect_breaches(wednesday, SLA_HOURS))

**Observe:** Wednesday's report contains Tuesday's ticket.

**Hypothesize:** the default list survives between calls.

**Inspect:** Python evaluates default values **once, when the `def` statement runs**, not on every call. (Recall Session 01: `def` is a statement executed at run time, and it creates a function object.) The `[]` becomes one list object stored on the function, and every call that uses the default gets that *same* list. You can see it:

In [ ]:
print(collect_breaches.__defaults__)

**Fix:** use `None` as the default, and create the mutable object inside the function, so each call gets a new one:

In [ ]:
def collect_breaches(tickets, sla_hours, found=None):
    if found is None:
        found = []
    for ticket in tickets:
        if ticket["age_hours"] > sla_hours[ticket["priority"]]:
            found.append(ticket["id"])
    return found


print("Tuesday:  ", collect_breaches(tuesday, SLA_HOURS))
print("Wednesday:", collect_breaches(wednesday, SLA_HOURS))

**Verify:** each call now starts empty.

The same applies to `{}`, `set()` and any other mutable default. Immutable defaults (`0`, `"open"`, `None`, a tuple) are always safe, because nobody can change them. Ruff flags mutable defaults with rule `B006`; you'll configure Ruff's rules in Session 04.

**Engineering Insight:** this isn't a quirk to memorise; it's the names-and-objects model again. One object, created once, reachable from every call.

## Part 3: Scope

Python looks names up in four places, in this order, often called **LEGB**: **L**ocal (the current function), **E**nclosing (outer functions, for nested functions), **G**lobal (the module), **B**uilt-in (`len`, `print` …).

Reading a module-level name from inside a function just works. **Assigning** to a name inside a function makes it local to that function for the *whole* function body, and that produces a confusing error.

**Predict:** what happens when this runs?

In [ ]:
import traceback

processed_count = 0


def process(ticket):
    processed_count += 1   # looks like it updates the module-level counter
    return ticket["id"]


try:
    process({"id": "INC-1"})
except UnboundLocalError:
    traceback.print_exc()

**Explain:** because `process` assigns to `processed_count`, Python treats that name as local throughout the function. `processed_count += 1` reads the local before anything has been assigned to it, so it fails.

You *could* fix this by writing `global processed_count` inside the function. Don't. Hidden module-level state is what made Wednesday's bug possible, and it makes functions hard to test. Pass values in and return results instead:

In [ ]:
def process_all(tickets):
    processed_ids = [ticket["id"] for ticket in tickets]
    return processed_ids, len(processed_ids)


ids, count = process_all([{"id": "INC-1"}, {"id": "INC-2"}])
print(ids, count)

Module-level **constants** such as `SLA_HOURS` are fine and normal; the convention is `UPPER_CASE` names that nobody reassigns. It's mutable module-level *state* that causes trouble.

## Part 4: Type hints

Type hints annotate what a function expects and returns:

In [ ]:
def oldest_open(tickets: list[dict], *, status: str = "open") -> int | None:
    """Age in hours of the oldest ticket with `status`, or None if there are none."""
    ages = [t["age_hours"] for t in tickets if t["status"] == status]
    return max(ages, default=None)


print(oldest_open(TEAM_TICKETS))
print(oldest_open(TEAM_TICKETS, status="resolved"))

The syntax you'll use most:

| Hint | Meaning |
|---|---|
| `int`, `str`, `bool`, `float` | That type |
| `list[dict]`, `dict[str, int]`, `tuple[str, int]`, `set[str]` | Collections and their contents |
| `-> None` | Returns nothing useful |
| `Path` (from `pathlib`) | Any class works as a hint |

For a value that may be missing, `int | None` means "an `int`, or `None`", like `Optional<Integer>` in Java or `int?` in C#. You'll see the older spelling `Optional[int]` in existing code; it means the same thing.

**Predict:** you learned in Session 01 that hints aren't enforced. What does this return?

In [ ]:
print(oldest_open([{"status": "open", "age_hours": "12"}]))
print(oldest_open.__annotations__)

**Explain:** it returns the string `'12'`, despite `-> int | None`. Python stores the hints in `__annotations__` and never checks them. They're for **people and tools**:

- Your editor (VS Code's Pylance) uses them for autocompletion and to underline obvious mistakes.
- Type checkers such as mypy or pyright check a whole codebase against them, much like a compiler would. We don't require one in this course, but many teams run one in CI.
- Libraries can *choose* to read them at run time. Pydantic (Session 06) and FastAPI (Session 07) do exactly that, turning hints into real validation.

| | Java / C# | TypeScript | Python |
|---|---|---|---|
| Where types are checked | Compiler, always | Compiler, erased at run time | Optional external tools; ignored by the interpreter |
| Wrong type at run time | Can't compile | Runs anyway | Runs anyway |
| External data checked? | No (deserialisers can) | No (zod etc. can) | No (Pydantic can) |

**Engineering Insight:** hints make code easier to read and refactor, so add them to every function others will call. But data from files and APIs must be **validated at run time**, which Part 7 does by hand and Session 06 does with Pydantic.

## Part 5: Exceptions

### `try`, `except`, `else`, `finally`

You know `try`/`catch`/`finally`. Python adds an `else` block that runs **only if no exception was raised**, which keeps the `try` block focused on the one line that can fail.

**Predict** the order of the printed lines for each input:

In [ ]:
def parse_age(raw):
    try:
        age = int(raw)
    except ValueError:
        print(f"  except:  {raw!r} is not a number")
        return None
    else:
        print(f"  else:    parsed {age}")
        return age
    finally:
        print("  finally: always runs, even after return")


for raw in ["26", "twenty"]:
    print(f"parse_age({raw!r}) -> {parse_age(raw)}")

**Explain:** `finally` runs on every path, including after `return`, which is why it's used for cleanup. The result line prints last because `parse_age` must finish, `finally` included, before `print` gets its argument.

### The built-in exceptions you'll actually meet

| Exception | Typical cause |
|---|---|
| `KeyError` | `d["missing"]` |
| `IndexError` | `items[99]` |
| `TypeError` | Wrong type for an operation: `"3" + 3`, calling with wrong arguments |
| `ValueError` | Right type, bad value: `int("twenty")` |
| `AttributeError` | `None.strip()`: often a `None` you didn't expect |
| `FileNotFoundError` | Opening a path that doesn't exist (a subclass of `OSError`) |
| `json.JSONDecodeError` | Malformed JSON (a subclass of `ValueError`) |
| `UnicodeDecodeError` | Reading bytes with the wrong text encoding |

They form a class hierarchy, and `except` matches subclasses too: `except ValueError` also catches `JSONDecodeError`, and `except OSError` also catches `FileNotFoundError`. Catch the **most specific** class that describes what you can handle.

### Ask forgiveness, or look before you leap?

Python culture leans towards **EAFP** ("easier to ask forgiveness than permission"): just try the operation and handle the failure. The alternative, **LBYL** ("look before you leap"), checks first:

In [ ]:
from pathlib import Path

config_path = Path("does-not-exist.json")

# LBYL: check, then act. Another process could delete the file between the two lines.
if config_path.exists():
    text = config_path.read_text(encoding="utf-8")
else:
    print("LBYL: no config file")

# EAFP: act, and handle the failure. No gap between check and use.
try:
    text = config_path.read_text(encoding="utf-8")
except FileNotFoundError:
    print("EAFP: no config file")

For files, EAFP is more correct: the check-then-act version has a race condition. For simple in-memory checks (`if key in d`), either is fine; choose whichever reads better.

### Raising, and your own exception types

Raise an exception when your function **can't do its job**. Give it a message that would help someone at 02:00: what failed, and on which input.

For errors that callers should be able to handle *specifically*, define your own exception class. It takes two lines. (Classes are Session 04's topic; for an exception, this is all you need.)

In [ ]:
class ExportError(Exception):
    """An export file could not be read or parsed."""


def check_export_name(file_name):
    if not file_name.endswith(".json"):
        raise ExportError(f"{file_name}: expected a .json export")
    return file_name


try:
    check_export_name("tickets-2026-10-21.csv")
except ExportError as error:
    print(f"{type(error).__name__}: {error}")

### Exception chaining, and how an exception travels

Often you catch a low-level exception and raise one that means something to *your* callers. `raise NewError(...) from error` does that while keeping the original as the **cause**, so no diagnostic detail is lost.

![How an exception travels up the call stack](../../../assets/diagrams/exception-propagation.svg)

**How this works**

- Calls go down the stack: `main` → `load_exports` → `read_export` → `json.load`. When `json.load` meets bad input, it raises `JSONDecodeError` and stops.
- Python looks for a matching `except` in the current frame, then the caller's frame, and so on upwards. Frames without a handler end immediately, but their `finally` blocks and `with` exits still run.
- `read_export` translates the error: it catches `JSONDecodeError` and raises `ExportError(...) from error`. Callers now need to know about one exception type, and the original is still attached as `__cause__`.
- `load_exports` is the right place to **handle** it, because it's the code that knows what to do: record the file as rejected and carry on with the next one. That's the fix for Tuesday's crash.
- If nothing handles an exception, it leaves `main`, Python prints a traceback and the process exits with code `1`. That's correct for a bug, and wrong for one bad input file among five.

Let's produce a real chained traceback and read it. The cell below raises through three functions and prints exactly what you'd see in a terminal:

In [ ]:
import json


def parse_export(text, file_name):
    try:
        return json.loads(text)
    except json.JSONDecodeError as error:
        raise ExportError(f"{file_name}: invalid JSON at line {error.lineno}") from error


def load_export(file_name, text):
    tickets = parse_export(text, file_name)
    return len(tickets)


def nightly_job():
    truncated = '[\n  {"id": "INC-1", "team": "platform"},\n  {"id": "INC-2", "team": '
    return load_export("tickets-2026-10-21.json", truncated)


try:
    nightly_job()
except ExportError:
    traceback.print_exc()

**How to read it:**

1. **Start at the very last line.** It's the exception that reached you: `ExportError: tickets-2026-10-21.json: invalid JSON at line 3`. Often that's enough.
2. **Read the frames above it, top to bottom**, as the path execution took: `nightly_job` called `load_export`, which called `parse_export`, which raised on the line shown. "Most recent call last" means the deepest frame is at the bottom.
3. **Look for "The above exception was the direct cause of the following exception".** Everything above that line is the *original* error, here the `JSONDecodeError` raised inside the `json` module, with its precise message.
4. **Find the first frame in *your* code.** Library frames are rarely where the bug is; your frame tells you which of your lines passed the bad input.

**Common Mistakes**, all of which destroy that information:

| Anti-pattern | Why it hurts | Instead |
|---|---|---|
| `except:` (bare) | Also catches `KeyboardInterrupt` and `SystemExit`; Ctrl+C stops working | Name the exception class |
| `except Exception: pass` | Errors vanish; the job "succeeds" with wrong output | Handle it, or let it propagate |
| `raise ExportError(...)` inside `except`, without `from` | The traceback says "During handling of the above exception, another exception occurred", which reads like a second bug | `raise ... from error` |
| `except Exception as e: print(e)` and carry on | The traceback and type are lost | Log the error with its traceback (Session 04+), or re-raise |
| One `try` around 50 lines | You can't tell which line failed, and you catch errors you didn't anticipate | Wrap only the lines that can raise what you're handling |

**Comparison with Java:** Python has no checked exceptions. Nothing forces you to declare or catch anything, so it's up to you to document what a function raises (in the docstring) and to decide deliberately where each kind of failure is handled.

## Part 6: Files and context managers

### `pathlib`

`pathlib.Path` is the modern way to work with file paths. It handles `/` versus `\` for you, and most file operations are methods on the path itself.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as workdir:
    exports_dir = Path(workdir) / "data" / "exports"          # / joins path parts
    exports_dir.mkdir(parents=True, exist_ok=True)

    report = exports_dir / "tickets-2026-10-21.json"
    report.write_text('[{"id": "INC-1"}]', encoding="utf-8")
    (exports_dir / "notes.txt").write_text("not an export", encoding="utf-8")

    print("name:  ", report.name)
    print("stem:  ", report.stem)
    print("suffix:", report.suffix)
    print("parent:", report.parent.name)
    print("exists:", report.exists(), report.is_file())
    print("glob:  ", sorted(p.name for p in exports_dir.glob("*.json")))
    print("text:  ", report.read_text(encoding="utf-8"))

`Path.read_text()` and `write_text()` are perfect for small files. For bigger ones, or when you need more control, you open the file yourself.

### `with`: guaranteed cleanup

`open()` returns a file object that holds an operating-system resource. It must be closed, **even if an exception happens while you're using it**. The `with` statement guarantees that:

In [ ]:
with tempfile.TemporaryDirectory() as workdir:
    path = Path(workdir) / "tickets.json"
    path.write_text("[]", encoding="utf-8")

    try:
        with open(path, encoding="utf-8") as file:
            first_line = file.readline()
            raise ValueError("something went wrong while processing")
    except ValueError as error:
        print("Caught:", error)

    print("File closed afterwards?", file.closed)

**Explain:** the exception left the `with` block early, yet the file was closed. `with` works with any **context manager**, an object that defines setup and teardown. Files, locks, database connections, HTTP clients (Session 05) and temporary directories (used throughout this notebook) are all context managers.

| Java | C# | Python |
|---|---|---|
| `try (var r = new FileReader(f)) { ... }` | `using var r = new StreamReader(f);` | `with open(f, encoding="utf-8") as r:` |

### Always pass `encoding="utf-8"`

Without an `encoding` argument, `open()` and `read_text()` use your **operating system's** default. On macOS and Linux that's almost always UTF-8. On Windows, for Python 3.13, it's usually a legacy code page such as cp1252. The same code then reads the same file differently on different machines.

**Predict:** the file is written as UTF-8. What does reading it as cp1252, the Windows default, produce?

In [ ]:
import locale

with tempfile.TemporaryDirectory() as workdir:
    path = Path(workdir) / "ticket.json"
    path.write_text('{"title": "Badge readers offline at the Zürich depot"}', encoding="utf-8")

    print("utf-8: ", path.read_text(encoding="utf-8"))
    print("cp1252:", path.read_text(encoding="cp1252"))

print("This machine's default:", locale.getpreferredencoding(False))

**Explain:** no error, just **mojibake**: `ü` became `Ã¼`, because its two UTF-8 bytes were read as two cp1252 characters. That silently corrupts data. (Some byte sequences are invalid in a given encoding and raise `UnicodeDecodeError` instead, which is at least loud.) JSON is UTF-8 by specification, so always write `encoding="utf-8"` explicitly; the course code always does. (Python 3.15 makes UTF-8 the default on every platform, as described in PEP 686. This course runs on 3.13, and code you write will often run on older versions, so keep passing it explicitly.)

To process a large file without loading it all into memory, iterate over the file object line by line: `for line in file:`. That's how you'd handle a multi-gigabyte log; for JSON, which must be parsed whole, the file needs to fit in memory.

## Part 7: JSON

The `json` module has four functions: `loads`/`dumps` work with strings, and `load`/`dump` work with open files.

In [ ]:
raw = (
    '{"id": "INC-1042", "priority": "P1", "age_hours": 7,'
    ' "assignee": null, "escalated": false, "tags": ["dns"]}'
)

ticket = json.loads(raw)
print(type(ticket).__name__, ticket)
print(repr(ticket["assignee"]), ticket["escalated"], type(ticket["age_hours"]).__name__)

print(json.dumps(ticket))
print(json.dumps(ticket, indent=2))

JSON maps onto Python types as follows:

| JSON | Python |
|---|---|
| object `{}` | `dict` |
| array `[]` | `list` |
| string | `str` |
| number | `int` or `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

The mapping is not a perfect round trip. **Predict** what each line does:

In [ ]:
from datetime import UTC, datetime

print(json.dumps({"pair": ("platform", "P1")}))   # tuple
print(json.dumps({1: "P1", 2: "P2"}))             # int keys
print(json.dumps({"team": "Zürich depot"}))        # non-ASCII text
print(json.dumps({"team": "Zürich depot"}, ensure_ascii=False))

for value in [{"tags": {"dns", "network"}}, {"at": datetime(2026, 10, 21, 2, 0, tzinfo=UTC)}]:
    try:
        json.dumps(value)
    except TypeError as error:
        print("TypeError:", error)

**Explain:**

- Tuples become arrays, so they come back as **lists** after a round trip.
- JSON object keys are always strings, so `1` comes back as `"1"`. A dict keyed by `(team, priority)` tuples can't be serialised at all; turn it into a list of records first.
- By default non-ASCII characters are escaped (`\u00fc`). That's valid JSON, but `ensure_ascii=False` keeps files readable (write them as UTF-8, as in Part 6).
- Sets and datetimes aren't JSON types. Convert them yourself: `sorted(tags)` for a set, `moment.isoformat()` for a datetime, which produces the standard `2026-10-21T02:00:00+00:00` format.

### Malformed JSON

`json.JSONDecodeError` tells you exactly where parsing failed. Use its attributes in your error messages:

In [ ]:
broken = '[\n  {"id": "INC-1"}\n  {"id": "INC-2"}\n]'

try:
    json.loads(broken)
except json.JSONDecodeError as error:
    print("msg:   ", error.msg)
    print("lineno:", error.lineno, "colno:", error.colno)

### The whole pipeline

Parsing successfully is only half the job. `json.load` happily returns a `dict` when you expected a `list`, or a record with `"age_hours": "12"`. Robust loading has several stages, and each can fail differently:

![Processing a JSON export safely](../../../assets/diagrams/json-processing-pipeline.svg)

**How this works**

- Reading and parsing can fail for the **whole file**: it's missing, it isn't UTF-8, or it isn't valid JSON. Those failures are translated into one `ExportError` (Part 5), and the file is skipped.
- Valid JSON with the wrong **shape** (an object where you expected a list of tickets) is also a whole-file problem, and needs its own check, because `json` can't know what you expected.
- **Records** are validated one at a time. A bad record shouldn't throw away the forty good ones in the same file, so problems are collected into a rejected list instead of raised.
- Only records that passed validation reach the transformation code from Session 02. That code can then use `ticket["age_hours"]` without defensive checks everywhere.
- Rejections are part of the output, not just a log line, so someone can fix the source data. The workbook builds exactly this pipeline.

### Validating a record by hand

Session 06 replaces this with Pydantic, but writing it once by hand shows you what any validator must do. The pattern: **collect all problems, rather than stopping at the first**, so one run tells the data owner everything that's wrong.

In [ ]:
FIELD_TYPES = {"id": str, "priority": str, "age_hours": int}


def record_problems(record):
    """Return a list of problems with one ticket record; an empty list means it's valid."""
    if not isinstance(record, dict):
        return ["record is not a JSON object"]
    problems = []
    for field, expected_type in FIELD_TYPES.items():
        if field not in record:
            problems.append(f"missing field {field!r}")
        elif not isinstance(record[field], expected_type):
            actual = type(record[field]).__name__
            problems.append(f"field {field!r} must be {expected_type.__name__}, got {actual}")
    return problems


for candidate in [
    {"id": "INC-1", "priority": "P1", "age_hours": 7},
    {"id": "INC-2", "priority": "P1", "age_hours": "12"},
    {"priority": 1},
    ["not", "a", "ticket"],
    {"id": "INC-3", "priority": "P2", "age_hours": True},
]:
    print(record_problems(candidate))

**Observe** the last result: `{"age_hours": true}` passed as a valid integer. `bool` is a subclass of `int` (Session 02, Part 3), so `isinstance(True, int)` is `True`. A JSON `true` in a numeric field is almost certainly a data error.

**Modify:** change `record_problems` so that booleans are rejected for `int` fields. (Hint: `isinstance(value, bool)` is the check you need, and its position in the `if`/`elif` chain matters.) Then add a check that `age_hours` isn't negative, which should only run when the value **is** an integer.

In [ ]:
# Your improved record_problems goes here. Re-run the loop above against it.

## Part 8: Debugging lab, the 02:00 crash

Here is a simplified version of the nightly job. It reads every export in a folder and counts tickets. One file is truncated, as in the engineering problem.

Run it, then work through **Observe → Hypothesize → Inspect → Fix → Verify**.

In [ ]:
def write_sample_exports(folder):
    """Create three small export files, one of them truncated."""
    exports = {
        "tickets-2026-10-19.json": '[{"id": "INC-1"}, {"id": "INC-2"}]',
        "tickets-2026-10-20.json": '[{"id": "INC-3"}, {"id": "INC-4"',
        "tickets-2026-10-21.json": '[{"id": "INC-5"}]',
    }
    for name, text in exports.items():
        (folder / name).write_text(text, encoding="utf-8")


def count_tickets(folder):
    total = 0
    for path in sorted(folder.glob("*.json")):
        with open(path, encoding="utf-8") as file:
            total += len(json.load(file))
    return total


with tempfile.TemporaryDirectory() as workdir:
    folder = Path(workdir)
    write_sample_exports(folder)
    try:
        print("Total tickets:", count_tickets(folder))
    except json.JSONDecodeError:
        traceback.print_exc()

**Observe:** read the traceback from the last line up. Which exception? Raised from which of *your* lines? Which file was being read? (The traceback doesn't say. That's part of the problem.)

<details>
<summary>Hint 1: what's wrong with the error itself?</summary>

`JSONDecodeError: Expecting ',' delimiter: line 1 column 33` gives a position but not the **file name**. At 02:00, with forty files, that's not enough. Translate it into an `ExportError` whose message names the file, and chain it with `from error` so the original detail is kept.

</details>

<details>
<summary>Hint 2: where should it be handled?</summary>

The failure of one file shouldn't stop the others. Inside the loop, wrap only the read-and-parse step in `try`/`except`, record the file name and reason in a `rejected` list, and `continue`. Return both the total and the rejected list (a tuple), so the caller can report them.

</details>

<details>
<summary>Hint 3: what should the caller see?</summary>

`count_tickets` should return a total of **3** (two tickets from the first file, one from the third) together with one rejection. **Verify** that the total is 3, that the rejected list names `tickets-2026-10-20.json`, and that no traceback is printed.

</details>

Write your fixed `count_tickets` below:

In [ ]:
# Your fixed count_tickets goes here. Test it inside a TemporaryDirectory, as above.

## Common mistakes

| Mistake | What happens | Instead |
|---|---|---|
| A mutable default (`items=[]`, `options={}`) | State leaks between calls | Default to `None`; create the object inside the function |
| Many positional options: `f(data, True, 24, "open")` | Unreadable, easy to swap arguments | Keyword-only parameters after `*` |
| Assigning to a module-level name inside a function | `UnboundLocalError` | Pass values in, return results |
| Trusting type hints to reject bad input | Wrong types flow through silently | Validate external data at run time |
| `except:` or `except Exception: pass` | Errors vanish, or Ctrl+C stops working | Catch specific exceptions; handle or re-raise |
| Raising a new exception inside `except` without `from` | Misleading traceback, cause hidden | `raise NewError(...) from error` |
| One bad input file stops the whole batch | No output at all | Handle per-file errors in the loop that owns the batch |
| `open(path)` without `with` | File left open if an exception occurs | `with open(path, encoding="utf-8") as file:` |
| No `encoding=` | Different results on Windows and Linux | Always `encoding="utf-8"` for text and JSON |
| Assuming parsed JSON has the expected shape | `TypeError`/`KeyError` deep inside later code | Check the shape and validate records right after parsing |
| `isinstance(value, int)` for a JSON number | `true`/`false` accepted as numbers | Reject `bool` explicitly |

## Practical engineering application

The pattern from this notebook appears in every integration you'll build: **parse at the edge, validate immediately, translate low-level errors into domain errors, and decide deliberately where each failure is handled.**

- In Session 05, HTTP errors and timeouts become your API client's own exceptions.
- In Session 06, Pydantic replaces hand-written `record_problems` functions.
- In Session 09, an LLM's "JSON" output gets exactly this treatment, because it's external data that's sometimes malformed.

## Key takeaways

- Arguments bind to the caller's objects; functions return one object (often a tuple); functions are objects themselves.
- Use keyword-only parameters (`*`) for options. Never use a mutable default; use `None` and create the object inside.
- Python resolves names Local → Enclosing → Global → Built-in. Assigning makes a name local. Avoid mutable global state.
- Type hints document and power tools, but the interpreter ignores them. Validate external data at run time.
- Catch specific exceptions, as close as possible to code that knows how to recover. Chain with `raise ... from error`.
- Read tracebacks from the last line up, then trace the frames down to the first line of your own code.
- `with` guarantees cleanup. `pathlib` handles paths. Always pass `encoding="utf-8"`.
- JSON gives you dicts, lists and primitives with no guarantees about shape. Check the shape, validate each record, and collect problems instead of crashing.

## Further reading

The curated references for this session are in [`../resources/references.md`](../resources/references.md).

**Next:** test yourself with **`02-practice-quiz.ipynb`**, then start the workbook in [`../workbook/assignment.md`](../workbook/assignment.md).